# RetinaPlus — External Validation on DDR

ประเมินโมเดล (ensemble v3+v4 EfficientNet-B4 ordinal) บนชุด **DDR** (~13,600 ภาพ) แบบ external
โดยใช้ preprocess / TTA / threshold เดียวกับ `app.py` เป๊ะ — **ไม่เทรนใหม่** แค่ inference

**ก่อนรัน:**
1. Settings → Accelerator → **GPU**
2. **+ Add Input** → แท็บ **Notebook** เพิ่มทั้ง 2 อัน: `notebookd5eec337a3` และ `notebookdec08e9df6` (แต่ละอันมี `best.pt`)
3. **+ Add Input** → แท็บ **Datasets** เพิ่ม `ddrdataset`
4. **Run All** (CELL 2 จะ auto-หา `.pt` + `DR_grading.csv` ให้เอง)


In [ ]:
# ============================ CELL 1 — setup ============================
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "timm"], check=False)

import os, glob, cv2, collections, numpy as np, pandas as pd
import torch, torch.nn as nn, timm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (cohen_kappa_score, accuracy_score, confusion_matrix,
                             classification_report, roc_auc_score)
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)

In [ ]:
# ============================ CELL 2 — config ============================
IMG_SIZE = 380
BACKBONE = "efficientnet_b4"
CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative"]

# ค่า OptimizedRounder ตอนเทรน — ห้าม fit ใหม่บน external
ROUNDER_COEFS = [0.515, 1.264, 2.376, 3.079]
REFERABLE_THR = ROUNDER_COEFS[1]          # grade >= 2 = referable DR

# --- weights v3/v4 (โมเดล 2 ตัว ชื่อ best.pt คนละ notebook) ---
# ⚠️ ก่อนรัน: + Add Input → Notebook → เพิ่มทั้ง 2 notebook นี้ก่อน
WEIGHT_PATHS = [
    "/kaggle/input/notebookd5eec337a3/best.pt",   # โมเดลตัวที่ 1 (v3)
    "/kaggle/input/notebookdec08e9df6/best.pt",   # โมเดลตัวที่ 2 (v4)
]
# เผื่อ slug เพี้ยน — auto-หา .pt ทุกตัวใน input ให้ด้วย
if not all(os.path.exists(p) for p in WEIGHT_PATHS):
    found = sorted(glob.glob("/kaggle/input/**/*.pt", recursive=True))
    print("auto-found .pt:", found)
    if found:
        WEIGHT_PATHS = found
print("WEIGHT_PATHS:", WEIGHT_PATHS)

# --- DDR dataset (mariaherrerot/ddrdataset) ---
CSV_PATH = "/kaggle/input/ddrdataset/DR_grading.csv"
if not os.path.exists(CSV_PATH):
    hits = glob.glob("/kaggle/input/**/DR_grading.csv", recursive=True)
    if hits:
        CSV_PATH = hits[0]
DATASET_ROOT = os.path.dirname(CSV_PATH)
NAME_COL  = "id_code"
LABEL_COL = "diagnosis"
DROP_LABELS = {5}                          # 5 = ungradable -> ตัดออก
print("CSV_PATH:", CSV_PATH)

# inference
BATCH    = 64
WORKERS  = 4
USE_FP16 = True                            # autocast เร็วขึ้น ~2x (QWK/Acc แทบไม่เปลี่ยน)
OUT_TAG  = "ddr"

In [ ]:
# ============================ CELL 3 — load ensemble (v3 + v4) ============================
def build_model():
    m = timm.create_model(BACKBONE, pretrained=False, num_classes=1,
                          drop_rate=0.3, drop_path_rate=0.2)
    return m.to(DEVICE).eval()

MEMBERS = []
for p in WEIGHT_PATHS:
    assert os.path.exists(p), f"ไม่พบ weight: {p}"
    m = build_model()
    sd = torch.load(p, map_location=DEVICE)
    if isinstance(sd, dict) and "state_dict" in sd:
        sd = sd["state_dict"]
    m.load_state_dict(sd)
    MEMBERS.append(m)
    print("loaded", os.path.basename(p))
print("ensemble size =", len(MEMBERS))
assert len(MEMBERS) >= 1, "ไม่มีโมเดลถูกโหลด — แก้ WEIGHT_PATHS ใน CELL 2"

In [ ]:
# ============================ CELL 4 — preprocess (เหมือน app.py เป๊ะ) ============================
_MEAN = np.array([0.485, 0.456, 0.406]); _STD = np.array([0.229, 0.224, 0.225])

def _crop_fundus(img, tol=7):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY); mask = g > tol
    if mask.sum() == 0: return img
    c = np.argwhere(mask); y0, x0 = c.min(0); y1, x1 = c.max(0) + 1
    return img[y0:y1, x0:x1]

def _clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB); l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(2.0, (8, 8)).apply(l)
    return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)

def preprocess(bgr):
    img = _crop_fundus(bgr)
    if img.size == 0: img = bgr
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    img = _clahe(img)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)         # RGB uint8

def to_chw(rgb):
    return ((rgb / 255.0 - _MEAN) / _STD).transpose(2, 0, 1).astype(np.float32)

In [ ]:
# ============================ CELL 5a — ส่องโครงสร้าง (รันดูครั้งเดียว) ============================
for r, d, f in os.walk(DATASET_ROOT):
    print(r, f"-> {len(f)} files", f[:2])
_df = pd.read_csv(CSV_PATH)
print("\ncolumns:", _df.columns.tolist())
print(_df.head())
print("\nนับ label:\n", _df[LABEL_COL].value_counts().sort_index())

In [ ]:
# ============================ CELL 5b — รวม items + Dataset + loader ============================
df = pd.read_csv(CSV_PATH)

# index: ชื่อไฟล์ (มี/ไม่มีนามสกุล) -> path เต็ม  (ทนทุกโครงสร้างโฟลเดอร์)
index = {}
for r, _, fs in os.walk(DATASET_ROOT):
    for fn in fs:
        if fn.lower().endswith((".jpg", ".jpeg", ".png")):
            full = os.path.join(r, fn)
            index[fn] = full
            index[os.path.splitext(fn)[0]] = full

items, miss = [], 0
for _, row in df.iterrows():
    lab = int(float(row[LABEL_COL]))
    if lab in DROP_LABELS:
        continue
    key = str(row[NAME_COL]).strip()
    p = index.get(key) or index.get(os.path.splitext(key)[0])
    if p is None:
        miss += 1; continue
    items.append((p, lab))

print(f"ใช้ได้ {len(items)} ภาพ | หาไฟล์ไม่เจอ {miss}")
print("การกระจาย label:", dict(sorted(collections.Counter(l for _, l in items).items())))
assert len(items) > 0, "ได้ 0 ภาพ — เช็ค path/คอลัมน์จาก CELL 5a"

class DRDataset(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        path, lab = self.items[i]
        bgr = cv2.imread(path, cv2.IMREAD_COLOR)
        if bgr is None:
            bgr = np.zeros((IMG_SIZE, IMG_SIZE, 3), np.uint8)
        return torch.from_numpy(to_chw(preprocess(bgr))), lab

loader = DataLoader(DRDataset(items), batch_size=BATCH, shuffle=False,
                    num_workers=WORKERS, pin_memory=True)

In [ ]:
# ============================ CELL 6 — inference (ensemble + TTA hflip/vflip) ============================
@torch.no_grad()
def ensemble_batch(x):
    x = x.to(DEVICE, non_blocking=True)
    total = torch.zeros(x.size(0), device=DEVICE)
    with torch.cuda.amp.autocast(enabled=USE_FP16):
        for m in MEMBERS:
            o = m(x).float().squeeze(1)
            o = o + m(torch.flip(x, [3])).float().squeeze(1) \
                  + m(torch.flip(x, [2])).float().squeeze(1)
            total += o / 3
    return (total / len(MEMBERS)).cpu().numpy()

scores, y_true = [], []
for bi, (x, lab) in enumerate(loader):
    scores.append(ensemble_batch(x))
    y_true.append(lab.numpy())
    if bi % 20 == 0:
        print(f"  batch {bi}/{len(loader)}")
scores = np.concatenate(scores)
y_true = np.concatenate(y_true)
y_pred = np.clip(np.digitize(scores, ROUNDER_COEFS), 0, 4)
print("เสร็จ:", len(scores), "ภาพ")

In [ ]:
# ============================ CELL 7 — metrics + confusion matrix ============================
qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
acc = accuracy_score(y_true, y_pred)
ref_true = (y_true >= 2).astype(int)            # referable = moderate ขึ้นไป
auc = roc_auc_score(ref_true, scores) if len(set(ref_true)) > 1 else float("nan")

print(f"\n===== EXTERNAL ({OUT_TAG}) — n = {len(y_true)} =====")
print(f"QWK (quadratic kappa) : {qwk:.4f}")
print(f"Accuracy              : {acc:.4f}")
print(f"Referable AUC (>=2)   : {auc:.4f}\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3, 4])
plt.figure(figsize=(6.2, 5.2))
plt.imshow(cm, cmap="Blues")
plt.title(f"External ({OUT_TAG})  QWK={qwk:.3f}  Acc={acc:.3f}  n={len(y_true)}")
plt.colorbar(fraction=0.046)
plt.xticks(range(5), CLASS_NAMES, rotation=40, ha="right")
plt.yticks(range(5), CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True")
thr = cm.max() / 2
for i in range(5):
    for j in range(5):
        plt.text(j, i, cm[i, j], ha="center",
                 color="white" if cm[i, j] > thr else "black", fontsize=11)
plt.tight_layout()
plt.savefig(f"confusion_external_{OUT_TAG}.png", dpi=150)
plt.show()

# เซฟผลรายภาพ + สรุป metric
pd.DataFrame({"path": [p for p, _ in items], "true": y_true,
              "pred": y_pred, "score": np.round(scores, 4)}
             ).to_csv(f"external_{OUT_TAG}_predictions.csv", index=False)
pd.DataFrame([{"dataset": OUT_TAG, "n": len(y_true), "QWK": qwk,
               "accuracy": acc, "referable_auc": auc}]
             ).to_csv(f"external_{OUT_TAG}_metrics.csv", index=False)
print("saved:",
      f"confusion_external_{OUT_TAG}.png,",
      f"external_{OUT_TAG}_predictions.csv,",
      f"external_{OUT_TAG}_metrics.csv")